# Lesson 7: backpropagation

Stage 7 - ONE training step in slow motion, on a fresh untrained model.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Normansrule/transparent-transformer-llm/blob/main/notebooks/07_backpropagation.ipynb) &nbsp; [lesson page](../stages/07_backpropagation/) &nbsp;|&nbsp; [live in your browser](https://Normansrule.github.io/transparent-transformer-llm/#7)

The output below was produced by the model saved in this repository. Run the cells to reproduce it, then change things.

In [1]:
# Setup: works on GitHub Codespaces, on your own machine, and on Google Colab (where it clones the repository first).
import os, sys
if not os.path.exists("transparent_transformer"):
    if os.path.exists("../transparent_transformer"):
        os.chdir("..")
    else:
        os.system("git clone -q https://github.com/Normansrule/transparent-transformer-llm.git")
        os.chdir("transparent-transformer-llm")
sys.path.insert(0, os.getcwd())
print("ready, working in", os.getcwd())

ready, working in /home/claude/transparent-transformer-llm


In [2]:
import numpy as np

from transparent_transformer import GPT, BPETokenizer, Config, paths
from transparent_transformer.loss import cross_entropy

tok = BPETokenizer.load(paths.TOKENIZER)
model = GPT(Config())                                  # random weights: it knows nothing
ids = tok.encode("The weather in Los Angeles is usually sunny and warm.")
x, y = np.array([ids[:-1]]), np.array([ids[1:]])

print("1. FORWARD   run the sentence through the model")
loss, dlogits = cross_entropy(model.forward(x), y)
print(f"             loss = {loss:.4f}\n")

print("2. BACKWARD  start from dLoss/dlogits = probabilities - one_hot(correct), walk back layer by layer")
model.backward(dlogits)
grads = model.gradients()
print(f"             {'weight matrix':<22}{'shape':<14}{'size of its gradient'}")
for name in ["ln_f.g", "block1.mlp.down.W", "block1.attn.qkv.W", "block0.mlp.down.W", "block0.attn.qkv.W", "embed.tok"]:
    g = grads[name]
    print(f"             {name:<22}{str(g.shape):<14}{np.linalg.norm(g):.4f}")

print("\n3. CHECK     is the hand-written calculus right? Wiggle ONE weight and measure the loss directly")
name, idx, h = "block0.mlp.up.W", (3, 7), 1e-2
w = model.parameters()[name]
old = w[idx]
w[idx] = old + h; up, _ = cross_entropy(model.forward(x), y)
w[idx] = old - h; down, _ = cross_entropy(model.forward(x), y)
w[idx] = old
print(f"             wiggle estimate  (loss(w+h) - loss(w-h)) / 2h = {(up - down) / (2 * h):+.6f}")
print(f"             backpropagation                                = {grads[name][idx]:+.6f}")
print("             same answer. Backprop gets ALL 100,000+ gradients from one backward pass;")
print("             wiggling would need two forward passes PER WEIGHT.\n")

print("4. UPDATE    w = w - learning_rate * gradient, for every weight")
model.forward(x); model.backward(dlogits)              # restore the caches for a clean update
for k, p in model.parameters().items():
    p -= 0.5 * model.gradients()[k]
after, _ = cross_entropy(model.forward(x), y)
print(f"             loss before {loss:.4f}  ->  after {after:.4f}")
print("\nThat drop is learning. Pretraining is this step, repeated 1,500 times on different text.")

1. FORWARD   run the sentence through the model
             loss = 6.6120

2. BACKWARD  start from dLoss/dlogits = probabilities - one_hot(correct), walk back layer by layer
             weight matrix         shape         size of its gradient
             ln_f.g                (64,)         0.0491
             block1.mlp.down.W     (256, 64)     1.9421
             block1.attn.qkv.W     (64, 192)     0.5292
             block0.mlp.down.W     (256, 64)     2.0766
             block0.attn.qkv.W     (64, 192)     0.5947
             embed.tok             (768, 64)     3.1129

3. CHECK     is the hand-written calculus right? Wiggle ONE weight and measure the loss directly
             wiggle estimate  (loss(w+h) - loss(w-h)) / 2h = -0.010611
             backpropagation                                = -0.010610
             same answer. Backprop gets ALL 100,000+ gradients from one backward pass;
             wiggling would need two forward passes PER WEIGHT.

4. UPDATE    w = w - learn

## Your turn

**Think first:** Why use backpropagation instead of wiggling each weight to see what happens?

Then open `classroom/exercises/ex07.py`, fill in the function, and run the cell below to grade it.

In [3]:
!python classroom/check.py | head -14

## Homework: 0 of 10 passed

| exercise | lesson | result |
|---|---|---|
| `ex01.py` | Input | ⬜ not started |
| `ex02.py` | Tokenization | ⬜ not started |
| `ex03.py` | Embedding | ⬜ not started |
| `ex04.py` | Transformer | ⬜ not started |
| `ex05.py` | Attention | ⬜ not started |
| `ex06.py` | Pretraining | ⬜ not started |
| `ex07.py` | Backpropagation | ⬜ not started |
| `ex08.py` | Alignment | ⬜ not started |
| `ex09.py` | Sampling | ⬜ not started |
| `ex10.py` | Output | ⬜ not started |
Traceback (most recent call last):
  File "/home/claude/transparent-transformer-llm/classroom/check.py", line 76, in <module>
    print(summary)
BrokenPipeError: [Errno 32] Broken pipe
